In [0]:
df = spark.sql("""select * from mis_automation.silver_caps.slv_vw_card_delinquency_history where snapshot_date>ACTIVATIONDATE and snapshot_date<add_months(ACTIVATIONDATE,12)""").toPandas()
print(df)

In [0]:
cols = ['snapshot_date','CRNNO','CREDITCARDNO','CARDCREATEDATE','ACTIVATIONDATE','DELINQUENCYBUCKET']

df[cols].head(2)

In [0]:
df[cols].groupby(['CREDITCARDNO','ACTIVATIONDATE']).agg(los=('snapshot_date',pd.Series.nunique)).reset_index()

In [0]:
df[cols].loc[df.CREDITCARDNO=="1531000000370110"].sort_values("snapshot_date")#"1531000004671703"

In [0]:
import seaborn
df_del = df[["CREDITCARDNO","snapshot_date",'DELINQUENCYBUCKET','ACTIVATIONDATE']].groupby(["CREDITCARDNO"]).agg(max_delinquencybucket=('DELINQUENCYBUCKET',max),min_activationdate=('ACTIVATIONDATE',min),max_snapshotdate=('snapshot_date',max)).reset_index()

In [0]:
df_del.head()

In [0]:
df_del["target_12m_date"] = df_del['min_activationdate']+pd.DateOffset(months=12)
df_del.head()


In [0]:
df_del.loc[df_del.target_12m_date < df_del.max_snapshotdate]

In [0]:
import matplotlib.pyplot as plt
df_del['max_delinquencybucket'] = pd.to_numeric(df_del['max_delinquencybucket'], errors='coerce').fillna(-1).astype(int)
df_del['max_delinquencybucket'].value_counts().sort_index()
df_del['max_delinquencybucket'].loc[df_del['max_delinquencybucket']>0].value_counts().sort_index().plot(kind="bar",figsize=(8,4))
plt.title("Distibution of Deliquency Buckets")
plt.xlabel("Bucket")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [0]:
!pip install --upgrade optuna statsmodels scikit-learn gputil dvc apache-airflow lightgbm xgboost catboost h2o python-tds



In [0]:
%restart_python